# PoBot — QLoRA Fine-Tuning (Colab)

Fine-tunes a small open model (default **Llama-3.2-1B-Instruct**) to answer
Hong Kong labour questions in PoBot's grounded, cited style, using the dataset
built by `build_dataset.py`.

**Runtime:** Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

**Honest framing:** this is *knowledge distillation* from the Groq 70B teacher
and a demonstration of the PEFT/LoRA skill + a cheap self-hostable model. A 1B
student won't beat the 70B on hard queries; its value is format discipline
(grounding, citations, refusal) and low-cost/offline deployment.

All hyperparameters live in `finetuning/config.yaml` (single source of truth).


## 1. Check the GPU


In [1]:
!nvidia-smi


Thu Jul  9 18:43:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install pinned dependencies


In [2]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes trl pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 73.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.8 MB/s eta 0:00:00:00:0100:01


## 3. Get the repo (dataset + config)

Replace the URL with your repo. The dataset (`finetuning/data/*.jsonl`) and
`config.yaml` are versioned there, so this is reproducible.


In [ ]:
# Option A: clone your repo
!git clone https://github.com/uzairahim/PoBot_RAG-Chatbot-for-Hong-Kong-Labour-Regulations.git pobot
%cd pobot/finetuning

# Option B (if not using git): upload train.jsonl, val.jsonl, config.yaml via the
# Files panel, then set the paths below accordingly.

## 4. Hugging Face auth

Llama-3.2 is a **gated** model — request access on its HF page, then add your
token to Colab **Secrets** (key icon, left sidebar) as `HF_TOKEN`.


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))


## 5. Load config


In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
BASE = cfg['base_model']
L, Q, T = cfg['lora'], cfg['quantization'], cfg['training']
print('base model:', BASE)
print('LoRA r/alpha:', L['r'], L['alpha'], '| epochs:', T['epochs'])


## 6. Load the dataset and render to text with the chat template


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ds = load_dataset('json', data_files={
    'train': cfg['dataset']['train_file'],
    'validation': cfg['dataset']['val_file'],
})

def to_text(ex):
    # Render our [{role,content}...] messages into the model's chat format.
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}

ds = ds.map(to_text, remove_columns=ds['train'].column_names)
print(ds)
print(ds['train'][0]['text'][:800])


## 7. Load the base model in 4-bit (QLoRA) + attach LoRA


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=Q['load_in_4bit'],
    bnb_4bit_quant_type=Q['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=Q['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=getattr(torch, Q['bnb_4bit_compute_dtype']),
)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')

lora = LoraConfig(
    r=L['r'], lora_alpha=L['alpha'], lora_dropout=L['dropout'],
    target_modules=L['target_modules'], task_type='CAUSAL_LM', bias='none',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()  # note the tiny % that is trainable


## 8. Train (SFTTrainer)


In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir='out',
    num_train_epochs=T['epochs'],
    learning_rate=T['learning_rate'],
    per_device_train_batch_size=T['per_device_batch_size'],
    gradient_accumulation_steps=T['gradient_accumulation_steps'],
    warmup_ratio=T['warmup_ratio'],
    max_seq_length=T['max_seq_length'],
    logging_steps=T['logging_steps'],
    save_steps=T['save_steps'],
    eval_strategy='epoch',
    dataset_text_field='text',
    bf16=True,
    report_to='none',
)
trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['validation'],
    processing_class=tok,
)
trainer.train()


## 9. Eyeball base vs fine-tuned

Qualitative check on a couple of validation prompts — look for grounding,
citations, and refusal behavior.


In [ ]:
import re
def generate(m, text):
    prompt = text.split('Answer:')[0] + 'Answer:'  # cut before the target answer
    ins = tok(prompt, return_tensors='pt').to(m.device)
    out = m.generate(**ins, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True).strip()

sample = ds['validation'][0]['text']
print('=== FINE-TUNED ===')
print(generate(model, sample))


## 10. Save the adapter

Saves the LoRA adapter (~10-50 MB), zips it, and downloads it. Commit it to
`finetuning/adapter/` in the repo, then run locally with `LLM_PROVIDER=hf_local`.


In [ ]:
adapter_dir = cfg['output']['adapter_dir']
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

import shutil
from google.colab import files
shutil.make_archive('pobot_adapter', 'zip', adapter_dir)
files.download('pobot_adapter.zip')

# Optional: push to the Hugging Face Hub instead of downloading
# model.push_to_hub('<you>/pobot-hk-labour-lora')


## Next steps

1. Unzip the adapter into `finetuning/adapter/` and commit it.
2. Locally: `pip install -r requirements-finetune.txt`.
3. In `.env`: `LLM_PROVIDER=hf_local` and `HF_ADAPTER_DIR=finetuning/adapter`.
4. `python chatbot.py` — the fine-tuned model now answers through the RAG pipeline.
